# Video Understanding con Qwen2.5-VL
### Descripción general + VideoQA desde dataset de Kaggle

Este notebook explora la comprensión de video con **Qwen2.5-VL-3B-Instruct**, una capacidad ausente en modelos como BLIP-2. El modelo recibe un video local y responde preguntas sobre su contenido en lenguaje natural.

---

### ¿Cómo procesa Qwen2.5-VL el video?

El video introduce una **dimensión temporal** que requiere innovaciones arquitectónicas específicas respecto al procesamiento de imágenes estáticas:

| Mecanismo | Descripción |
|---|---|
| **Dynamic FPS Sampling** | Extrae frames a una tasa configurable (`fps`). El modelo fue entrenado con distintas tasas para ser robusto a este parámetro. Menor `fps` = menos tokens = menor VRAM. |
| **mRoPE 3D** | Extiende el Rotary Position Embedding a 3 dimensiones: tiempo **(T)**, alto **(H)** y ancho **(W)**. Cada frame recibe IDs de posición temporal absolutos, permitiendo al modelo localizar eventos por segundo. |
| **3D Convolution en ViT** | El encoder visual usa convolución 3D para procesar parches espacio-temporales de múltiples frames como un volumen unificado. |
| **Window Attention** | Atención local dentro de ventanas de 112×112 px, reduciendo el costo cuadrático de la atención global sobre secuencias de video largas. |

> ⚠️ **Ajuste para T4/P100 de Kaggle:** Con `fps=1.0` y `max_pixels=360*420`, un video de 30 s genera ~30 frames → ~600 tokens visuales. Esto cabe cómodamente en 16 GB de VRAM. Para videos más largos reduce `fps` a `0.5`.

In [ ]:
!pip install -q transformers>=4.49.0 accelerate qwen-vl-utils bitsandbytes decord

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.9.0+cu126
True
12.6


# Carga del modelo

### `Qwen/Qwen2.5-VL-3B-Instruct`

| **Feature** | **Description** |
|---|---|
| **Arquitectura** | ViT dinámico 3D + Transformer Decoder (Qwen2.5) |
| **Parámetros** | ~3B |
| **Entrada** | Imagen, video o lista de frames + Texto |
| **Resolución por frame** | Dinámica (hasta 1280×1280 px) |
| **Duración de video** | Hasta 1 hora (variante 72B); minutos en variante 3B con GPU limitada |
| **Max tokens contexto** | 32,768 tokens |
| **Cuantización usada** | 4-bit (BitsAndBytes) → ~4 GB VRAM |
| **Licencia** | Apache 2.0 |
| **Repositorio** | [Qwen2.5-VL en Hugging Face](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct) |

In [ ]:
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from qwen_vl_utils import process_vision_info
from IPython.display import Video, display
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo disponible: {device}")

# Cuantización 4-bit: reduce VRAM de ~8 GB (fp16) a ~4 GB
# Permite correr cómodamente en T4/P100 de Kaggle
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f" Qwen2.5-VL cargado en: {next(model.parameters()).device}")

Dispositivo disponible: cuda


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

 Qwen2.5-VL cargado en: cuda:0


# Funciones del agente de video

In [ ]:
def ask_video(video_path, question, fps=1.0, max_pixels=360*420, max_new_tokens=256):
    """
    Responde una pregunta en lenguaje natural sobre un video local.

    El parámetro `fps` controla cuántos frames por segundo extrae el modelo.
    `process_vision_info` usa la librería `decord` internamente para decodificar
    el video y aplicar el muestreo dinámico de frames.

    A diferencia del procesamiento de imágenes (donde se pasa `images=` al
    processor), el video requiere `videos=` como argumento separado, reflejando
    que el modelo usa un encoder espacio-temporal distinto.

    Args:
        video_path     : ruta local al video (.mp4, .avi, .mov, .mkv)
        question       : pregunta en lenguaje natural
        fps            : frames por segundo a extraer (recomendado: 1.0 para T4)
        max_pixels     : resolución máxima por frame (recomendado: 360*420)
        max_new_tokens : máximo de tokens a generar en la respuesta

    Returns:
        str: respuesta generada por el modelo
    """
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_path,
                "fps": fps,
                "max_pixels": max_pixels
            },
            {"type": "text", "text": question}
        ]
    }]

    # Formatear prompt con los tokens especiales del chat de Qwen
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # process_vision_info extrae y preprocesa los frames del video
    # Retorna (image_inputs, video_inputs) — aquí solo usamos video_inputs
    _, video_inputs = process_vision_info(messages)

    # Nota: se usa `videos=` (no `images=`) para entrada de video
    inputs = processor(
        text=[text_prompt],
        videos=video_inputs,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False  # greedy: determinista y reproducible en clase
        )

    # Decodificar solo los tokens nuevos (excluir el prompt de entrada)
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.decode(generated_ids[0], skip_special_tokens=True)


def video_qa_session(video_path, questions, fps=1.0, max_pixels=360*420, max_new_tokens=256):
    """
    Sesión de preguntas múltiples sobre un mismo video.
    Muestra el video y ejecuta cada pregunta secuencialmente.

    Args:
        video_path     : ruta local al video
        questions      : lista de preguntas en lenguaje natural
        fps            : frames por segundo a extraer
        max_pixels     : resolución máxima por frame
        max_new_tokens : máximo de tokens por respuesta
    """
    display(Video(video_path, width=640))
    print("=" * 60)
    print(f"📹 Video  : {video_path}")
    print(f"⚙️  fps={fps}  |  max_pixels={max_pixels}")
    print("=" * 60)

    for i, question in enumerate(questions, 1):
        print(f"\n Pregunta {i}: {question}")
        answer = ask_video(video_path, question, fps=fps,
                           max_pixels=max_pixels, max_new_tokens=max_new_tokens)
        print(f" Respuesta {i}: {answer}")
        print("-" * 60)

# Descripción general del video

Primero pedimos al modelo una descripción libre del contenido completo.
Esto evalúa su capacidad de síntesis temporal: debe integrar información
de múltiples frames para producir una descripción coherente de lo que ocurre.

In [ ]:
# ── Configura aquí la ruta a tu video ────────────────────────────────────────
# Sube el video a tu dataset de Kaggle y ajusta la ruta:
VIDEO_PATH = "/kaggle/input/datasets/bryanyamacruz/video-2/WhatsApp Video 2026-03-13 at 11.50.25.mp4"  # <── modificar

# Mostrar el video en el notebook
display(Video(VIDEO_PATH, width=640))

# Descripción general
print("\n Descripción general del video:")
print("-" * 60)
description = ask_video(
    VIDEO_PATH,
    question="Describe in detail what is happening in this video. "
             "Include the setting, objects, people, actions, and how the scene evolves over time.",
    fps=0.5,
    max_new_tokens=400
)
print(description)


 Descripción general del video:
------------------------------------------------------------
The video begins with a wide shot of a mountainous landscape under a clear blue sky. The mountains are covered with lush green vegetation, and the valley below is also visible, showing a mix of greenery and some cultivated land.

As the video progresses, the camera remains stationary for several seconds, allowing viewers to take in the serene beauty of the natural scenery. The clouds in the sky are scattered but add depth to the landscape.

Suddenly, a large plume of white smoke or ash begins to rise from the top of one of the mountains. The smoke billows upwards rapidly, obscuring part of the mountain and creating a dramatic contrast against the blue sky.

The smoke continues to rise and spread outwards, gradually filling more of the sky and partially obscuring the view of the mountains and valley below. The scene captures the sudden change in the landscape due to the volcanic activity.

Over

# Descripción general del video

**Qwen 2.5 VL**

El modelo logró reconocer lo que pasa, es un video tranquilo donde se graba la montñaña y como las nubes se mueven, pero en la parte final evidenciamos que el modelo exageró un poco.


### Localización temporal:

In [ ]:
# ── Configura aquí la ruta a tu video ────────────────────────────────────────
# Sube el video a tu dataset de Kaggle y ajusta la ruta:
VIDEO_PATH = "/kaggle/input/datasets/bryanyamacruz/video-2/WhatsApp Video 2026-03-13 at 11.50.25.mp4"  # <── modificar

# Mostrar el video en el notebook
display(Video(VIDEO_PATH, width=640))

# Descripción general
print("\n🔍 Descripción general del video:")
print("-" * 60)
description = ask_video(
    VIDEO_PATH,
    question="At that moment the clouds appeared..",
    fps=0.5,
    max_new_tokens=400
)
print(description)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.



🔍 Descripción general del video:
------------------------------------------------------------
The video shows a landscape with mountains and valleys, with clouds in the sky. The clouds move across the sky, and the sun is shining on the mountains. The video ends with the clouds moving away from the mountains.


# Localización temporal
**Qwen 2.5 VL**

En este contexto el modelo logró reconocer el estado del video sin necesidad de exagerar el escenario.

###  cambio en la pregunta, más especifico lo del tiempo del vìdeo

In [ ]:


# ── Configura aquí la ruta a tu video ────────────────────────────────────────
# Sube el video a tu dataset de Kaggle y ajusta la ruta:
VIDEO_PATH = "/kaggle/input/datasets/bryanyamacruz/video-2/WhatsApp Video 2026-03-13 at 11.50.25.mp4"  # <── modificar

# Mostrar el video en el notebook
display(Video(VIDEO_PATH, width=640))

# Descripción general
print("\n🔍 Descripción general del video:")
print("-" * 60)
description = ask_video(
    VIDEO_PATH,
    question="Clouds appear in the video, but what color are the mountains?",
    fps=0.1,
    max_new_tokens=400
)
print(description)



🔍 Descripción general del video:
------------------------------------------------------------
The mountains in the video are green.


# Cambio en la pregunta, más especifico lo del tiempo del vìdeo

**Qwen 2.5 VL**

El modelo logró reconocer lo que pasa,se preguntó: "Clouds appear in the video, but what color are the mountains?"

La respuesta del modelo fue precisa, "The mountains in the video are green." > "Las montañas del vídeo son verdes.", quiza el cambio se notó ya que se modificó fps=0.1.

---
# Actividades

1. **Localización temporal**: Usa un video donde ocurra un evento específico en un momento identificable (ej: un gol, una explosión, un cambio de escena). Pregunta: *"At what moment in the video does [evento] occur? Describe the timestamp."* ¿El modelo identifica correctamente el momento?

2. **Trade-off fps vs. calidad**: Registra el tiempo de inferencia y evalúa la calidad de cada respuesta. ¿A partir de qué `fps` la calidad se estabiliza? ¿Qué implicaciones tiene esto para un sistema en producción con restricciones de latencia?

Esta prueba ilustra el **trade-off fundamental** en Video-LLMs: el parámetro `fps`
controla cuántos frames por segundo se extraen del video antes de procesarlos.

| `fps` | Frames (30s video) | Tokens visuales aprox. | Uso de VRAM | Detalle temporal |
|---|---|---|---|---|
| 0.5 | ~15 | ~300 | Mínimo | Bajo |
| 1.0 | ~30 | ~600 | Moderado | Medio |
| 2.0 | ~60 | ~1200 | Alto | Alto |

Observa cómo varía la calidad de la respuesta y el tiempo de inferencia.

In [ ]:
import time

VIDEO_PATH = "/kaggle/input/datasets/bryanyamacruz/explosion/WhatsApp Video 2026-03-13 at 12.04.16.mp4"

question_fps = "Describe step by stwhat happens in this video, in chronological order"
fps_values = [0.5, 1.0, 2.0]

print(f"Pregunta: {question_fps}")
print("=" * 60)

for fps_val in fps_values:
    start = time.time()
    answer = ask_video(VIDEO_PATH, question_fps, fps=fps_val, max_new_tokens=300)
    elapsed = time.time() - start

    print(f"\n⏱️  fps={fps_val}  |  Tiempo de inferencia: {elapsed:.1f}s")
    print(f"📝 Respuesta:\n{answer}")
    print("-" * 60)

Pregunta: Describe step by stwhat happens in this video, in chronological order

⏱️  fps=0.5  |  Tiempo de inferencia: 18.4s
📝 Respuesta:
The video begins with an aerial view of a cityscape, showing numerous buildings and streets. The camera pans across the area, providing a comprehensive overview of the urban environment.

As the camera continues to pan, it captures the bustling activity within the city. The buildings, varying in height and design, are densely packed, creating a sense of density and complexity. The streets are lined with trees and green spaces, adding a touch of nature to the urban landscape.

The camera then focuses on a specific building, highlighting its unique architectural features. This building stands out from the rest due to its distinctive design and prominent location within the city.

Throughout the video, the camera captures various angles and perspectives, showcasing the intricate details of the city's layout and architecture. The overall scene is one of 

# Video Nuevo  - *fps_values* - Tiempo de inferencia

**Qwen 2.5 VL**

El análisis de las tres pruebas (0.5, 1.0 y 2.0 fps)

* 0.5 fps (Latencia: 18.4s): La calidad es insuficiente. El modelo realiza una descripción genérica de un entorno urbano ("ciudad vibrante", "arquitectura única"), pero omite por completo el evento crítico

* 1.0 fps (Latencia: 18.0s): Es el punto óptimo. El modelo identifica el contexto temporal (atardecer) y, lo más importante, detecta y describe cronológicamente la explosión, el fireball y la columna de humo.

* 2.0 fps (Latencia: 19.4s): Se observa un fenómeno de degradación. Aunque el tiempo de procesamiento aumenta, la calidad de la respuesta empeora. El modelo se distrae con detalles estéticos (colores del cielo) y vuelve a omitir el evento de la explosión.